# 03 - Case Retrieval

Retrieval menggunakan TF-IDF dan cosine similarity. TF-IDF hanya dibangun dari train/case base untuk menghindari data leakage.

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path('..').resolve()
DATA_DIR = PROJECT_ROOT / 'data'
RAW_PDF_DIR = DATA_DIR / 'raw' / 'pdf'
RAW_TEXT_DIR = DATA_DIR / 'raw' / 'text'
PROCESSED_DIR = DATA_DIR / 'processed'
EVAL_DIR = DATA_DIR / 'eval'
RESULTS_DIR = DATA_DIR / 'results'

import joblib
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
train_df = pd.read_csv(PROCESSED_DIR / 'case_base_train.csv')
test_df = pd.read_csv(EVAL_DIR / 'test_cases.csv')
vectorizer = joblib.load(PROCESSED_DIR / 'tfidf_vectorizer.joblib')
tfidf_matrix = joblib.load(PROCESSED_DIR / 'tfidf_matrix.joblib')
print('Train:', len(train_df))
print('Test:', len(test_df))
print('TF-IDF matrix:', tfidf_matrix.shape)

Train: 32
Test: 8
TF-IDF matrix: (32, 5000)


In [5]:
def retrieve(query, k=5):
    query_vector = vectorizer.transform([str(query).lower()])
    similarities = cosine_similarity(query_vector, tfidf_matrix).flatten()
    top_indices = similarities.argsort()[::-1][:k]
    results = train_df.iloc[top_indices].copy()
    results['similarity_score'] = similarities[top_indices]
    return results[['case_id','no_perkara','terdakwa','pasal','solution_label','similarity_score']]

retrieve('pencurian dalam keadaan memberatkan mengambil barang milik orang lain', k=5)

,case_id,no_perkara,terdakwa,pasal,solution_label,similarity_score
4,case_005,118/Pid.B/2026/PN.Tng,Marsin Bin Osip,Pasal 54 ayat (1) KUHP,Pidana Penjara 7-12 Bulan,0.097868
22,case_027,1821/Pid.B/2024/PN.Tng,M. Erik Prasetiya Bin (alm) Edi,Pasal 363 ayat 1 ke 5 KUHPidana; pasal 363 aya...,Pidana Tidak Teridentifikasi,0.096281
9,case_013,1337/Pid.B/2025/PN.Tng,Sumaryanto Bin (Alm) Sukiran,Pasal 363 Ayat 1 ke-5 KUHP; Pasal 363 ayat (1)...,Pidana Penjara > 12 Bulan,0.067409
26,case_032,1892/Pid.B/2024/PN.Tng,KOMARUDDIN Alias UDIN Bin (Alm) ARIFIN,Pasal 363 ayat (1) ke- 3 dan ke-5 KUHPidana; P...,Pidana Penjara > 12 Bulan,0.065429
31,case_040,617/Pid.B/2026/PN.Tng,SEKAR DHANIE Ad (Alm) SULAMTO,Pasal 362 KUHP; Pasal 476 Undang Undang Nomor ...,Pidana Penjara > 12 Bulan,0.063313


In [6]:
pd.read_csv(RESULTS_DIR / 'retrieval_results_sample.csv').head(10)

,query_id,query,rank,case_id,no_perkara,solution_label,similarity_score
0,sample_001,pencurian dalam keadaan memberatkan mengambil ...,1,case_005,118/Pid.B/2026/PN.Tng,Pidana Penjara 7-12 Bulan,0.088420
1,sample_001,pencurian dalam keadaan memberatkan mengambil ...,2,case_027,1821/Pid.B/2024/PN.Tng,Pidana Tidak Teridentifikasi,0.088239
2,sample_001,pencurian dalam keadaan memberatkan mengambil ...,3,case_003,1184/Pid.B/2025/PN.Tng,Pidana Penjara > 12 Bulan,0.067646
3,sample_001,pencurian dalam keadaan memberatkan mengambil ...,4,case_040,617/Pid.B/2026/PN.Tng,Pidana Penjara > 12 Bulan,0.066342
4,sample_001,pencurian dalam keadaan memberatkan mengambil ...,5,case_032,1892/Pid.B/2024/PN.Tng,Pidana Penjara > 12 Bulan,0.064807
5,sample_002,terdakwa mengambil sepeda motor atau barang mi...,1,case_002,1134/Pid.B/2025/PN.Tng,Pidana Penjara > 12 Bulan,0.160493
6,sample_002,terdakwa mengambil sepeda motor atau barang mi...,2,case_027,1821/Pid.B/2024/PN.Tng,Pidana Tidak Teridentifikasi,0.153592
7,sample_002,terdakwa mengambil sepeda motor atau barang mi...,3,case_025,1800/Pid.B/2024/PN.Tng,Pidana Penjara > 12 Bulan,0.143247
8,sample_002,terdakwa mengambil sepeda motor atau barang mi...,4,case_038,465/Pid.B/2026/PN.Tng,Pidana Penjara > 12 Bulan,0.122193
9,sample_002,terdakwa mengambil sepeda motor atau barang mi...,5,case_036,1960/Pid.B/2025/PN.Tng,Pidana Penjara > 12 Bulan,0.119375
